<div align="center">
  <img src="assets/Day6.png" alt="Databricks 14 Days AI Challenge - Day 06" width="800"/>
</div>

## DAY 6 (14/01/26) – Medallion Architecture

### 📚 Learning Objectives
Today we implement the **Medallion Architecture**, the industry-standard design pattern for building data lakes. Instead of a messy swamp of files, we organize data into three distinct quality levels:
* **🥉 Bronze (Raw):** The landing zone. Data is ingested "as-is" from the source. We add metadata (ingestion time) but do not change the schema.
* **🥈 Silver (Cleaned):** The enterprise layer. Data is deduplicated, cleaned, and enriched. This is the "Single Source of Truth." 
* **🥇 Gold (Aggregated):** The business layer. Data is aggregated for specific use cases (Dashboards, ML). It is highly optimized for read performance.

### 🚀 Strategy: The Pipeline Design
1.  **Ingest (Bronze):** Load our raw CSV files and save them as a Delta table, adding an ingestion timestamp for auditing.
2.  **Refine (Silver):** Filter out bad data (e.g., negative prices), remove duplicates, and add derived columns like `price_tier`.
3.  **Aggregate (Gold):** Calculate "Product Performance" metrics (Views vs. Purchases) to power a Business Intelligence dashboard.

###The Bronze Layer (Raw Ingestion)
* **Task**: Ingest raw data into the Bronze Delta table. 
* **Concept**: The goal here is speed and fidelity. We want to get the data out of the "fragile" CSV format and into "robust" Delta format as quickly as possible.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, when, countDistinct, sum, lit

# Define Paths (Using our established Volume structure)
base_path = "/Volumes/workspace/ecommerce/ecommerce_data"
bronze_path = f"{base_path}/delta/bronze_events"
silver_path = f"{base_path}/delta/silver_events"
gold_path   = f"{base_path}/delta/gold_product_perf"

# 1. READ RAW DATA (Simulating Source System)
# We load both Oct and Nov files to build our full raw history
df_raw = spark.read.option("header", "true") \
                   .option("inferSchema", "true") \
                   .csv(f"{base_path}/2019-*.csv")

# 2. TRANSFORM: Add Audit Columns
# We add 'ingestion_ts' so we know exactly when this data entered our lake
df_bronze = df_raw.withColumn("ingestion_ts", current_timestamp())

# 3. WRITE TO BRONZE
# We use 'overwrite' here for the demo, but in prod, this would likely be 'append'
print(f"🥉 Writing {df_bronze.count():,} rows to Bronze Layer...")
df_bronze.write.format("delta").mode("overwrite").save(bronze_path)

print(f"✅ Bronze Layer successfully built at: {bronze_path}")

###The Silver Layer (Cleaning & Enrichment)
* **Task**: Clean data and add price_tier. 
* **Concept**: Silver data must be trustworthy. We enforce quality rules here (e.g., "Price must be positive").

In [0]:
# 1. READ FROM BRONZE
df_bronze_source = spark.read.format("delta").load(bronze_path)

# 2. APPLY CLEANING RULES
# - Filter out bad data (Price <= 0, extremely high outliers)
# - Remove duplicates based on session and time
df_cleaned = df_bronze_source \
    .filter((col("price") > 0) & (col("price") < 50000)) \
    .dropDuplicates(["user_session", "event_time", "product_id"])

# 3. ENRICHMENT (Derived Columns)
# - Add Date column for easier partitioning/querying
# - Add 'price_tier' for segmentation analysis
df_silver = df_cleaned \
    .withColumn("event_date", to_date("event_time")) \
    .withColumn("price_tier",
        when(col("price") < 50, "Budget")
        .when(col("price") < 300, "Mid-Range")
        .otherwise("Premium")
    )

# 4. WRITE TO SILVER
print(f"🥈 Writing {df_silver.count():,} cleaned rows to Silver Layer...")
df_silver.write.format("delta").mode("overwrite").save(silver_path)

# Validation
print("--- Silver Sample Data ---")
df_silver.select("event_time", "event_type", "price", "price_tier").show(5)

###The Gold Layer (Business Aggregates)
* **Task**: Create specific aggregations for reporting. 
* **Concept**: Gold tables are often "read-heavy." We pre-calculate complex metrics (like Conversion Rate) so the dashboard tools (PowerBI/Tableau) don't have to compute them on the fly.

In [0]:
# 1. READ FROM SILVER
df_silver_source = spark.read.format("delta").load(silver_path)

# 2. CALCULATE AGGREGATIONS
# We pivot logic to count views vs purchases per product
df_gold = df_silver_source.groupBy("product_id", "category_code", "brand") \
    .agg(
        # Count unique users who VIEWED
        countDistinct(when(col("event_type") == "view", col("user_id"))).alias("unique_views"),
        # Count unique users who PURCHASED
        countDistinct(when(col("event_type") == "purchase", col("user_id"))).alias("unique_purchases"),
        # Total Revenue
        sum(when(col("event_type") == "purchase", col("price"))).alias("total_revenue")
    )

# 3. ADD KPIS (Conversion Rate)
# Logic: Purchases / Views. Handle division by zero.
df_gold_final = df_gold.withColumn(
    "conversion_rate_pct", 
    (col("unique_purchases") / (col("unique_views") + 1)) * 100
).fillna(0)

# 4. WRITE TO GOLD
print(f"🥇 Writing Business Aggregates to Gold Layer...")
df_gold_final.write.format("delta").mode("overwrite").save(gold_path)

print(f"✅ Gold Layer successfully built at: {gold_path}")

###Visualization (Business Value)
* **Task**: Display the insights. 
* **Concept**: In Databricks, the display() command allows us to instantly visualize the Gold data, proving its value to stakeholders.

In [0]:
# Load Gold Data
gold_data = spark.read.format("delta").load(gold_path)

# Visualize Top Performing Products by Revenue
# (Click the 'Plot' icon in the output below to visualize this as a Bar Chart)
display(
    gold_data
    .filter(col("total_revenue") > 1000)
    .orderBy(col("total_revenue").desc())
    .limit(10)
)

### 🧠 Key Learnings & Takeaways
* **Separation of Concerns:** By splitting the pipeline, we ensured that raw data is always available for debugging (Bronze), while analysts get clean data (Silver) without needing to clean it themselves.
* **Incremental Quality:** Each layer adds value—Bronze (Ingestion Speed) -> Silver (Trust/Quality) -> Gold (Insight/Speed).
* **Efficient Aggregation:** We performed the heavy computation (Conversion Rate) in the Gold layer job, so the end-user's dashboard loads instantly.
* **Delta Everywhere:** Using Delta format across all layers ensures ACID compliance and versioning throughout the entire pipeline.